# Interactive small-model pilot
One normal interactive Colab session, max 100 steps / 120 seconds. Fresh host-verified data and safe-tensor weights. CPU by default; optionally select CUDA in the cell. No distributed worker farm, keepalive, account rotation or paid API. Host validation is required and this does not replace the master.


In [ ]:
import subprocess, sys, json, time, hashlib, zipfile
from pathlib import Path
from google.colab import files
uploaded = files.upload()  # Select the proposal ZIP exported by your Debian mission.
assert len(uploaded) == 1
package = Path(next(iter(uploaded)))
with zipfile.ZipFile(package) as archive:
    assert set(archive.namelist()) == {'job.json', 'pool.jsonl'}
    assert all(i.file_size < 2**20 for i in archive.infolist())
    job = json.loads(archive.read('job.json'))
    pool_bytes = archive.read('pool.jsonl')
assert hashlib.sha256(pool_bytes).hexdigest() == job['pool_sha256']
revision = job['source_revision']
assert len(revision) == 40 and all(c in '0123456789abcdef' for c in revision)
lab = Path('/content/v100-interactive-pilot')
if not (lab / '.git').exists():
    subprocess.run(['git','clone','--no-checkout','https://github.com/Mantiee/rlm.git',str(lab)],check=True)
subprocess.run(['git','-C',str(lab),'fetch','origin',revision],check=True)
subprocess.run(['git','-C',str(lab),'checkout','--detach',revision],check=True)
print('PINNED SOURCE:', revision, 'JOB:', job['job_id'])
sys.path.insert(0,str(lab))
import torch
from rlm.v100.insights import verify_record
rows = [json.loads(line) for line in pool_bytes.decode().splitlines() if line]
for row in rows: verify_record(row)
print('Torch:', torch.__version__)


In [ ]:
from rlm.v100.compute_kernel import run, validate
import rlm.v100.compute_kernel as kernel_module
DEVICE = "cpu"  # Optional "cuda" in an interactive GPU notebook.
experiment = job["experiment"]
validate(experiment)
assert hashlib.sha256(Path(kernel_module.__file__).read_bytes()).hexdigest() == experiment["kernel_sha256"]
output = Path('/content/v100-interactive-result')
assert not output.exists(), "Use a new output directory for each experiment."
report = run(experiment, output, device=DEVICE)
print(json.dumps(report, indent=2))


In [ ]:
weights = output / "weights.safetensors"
manifest = {k: job[k] for k in ("job_id", "source_revision", "pool_sha256")}
manifest["weights_sha256"] = hashlib.sha256(weights.read_bytes()).hexdigest()
result_zip = Path('/content/v100-notebook-result.zip')
with zipfile.ZipFile(result_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.writestr('manifest.json', json.dumps(manifest))
    archive.writestr('metrics.json', json.dumps(report["metrics"], allow_nan=False))
    archive.write(weights, 'weights.safetensors')
files.download(str(result_zip))
print("Import on Debian with colab-import; independent local validation remains mandatory.")
